In [39]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor

df = pd.read_excel("./COMBINED C-PEP _ ISR.xlsx", sheet_name="combined")
def create_lag_features(df, lags=3):
    df_lag = df.copy()
    for lag in range(1, lags + 1):
        df_lag[f"CPeptide_lag{lag}"] = df_lag.groupby("Sample ID")["C-Peptide"].shift(lag)
    return df_lag.dropna()

df_lagged = create_lag_features(df)

X_rf = df_lagged[['Time (min)', 'C-Peptide', 'CPeptide_lag1', 'CPeptide_lag2', "CPeptide_lag3"]]
y_rf = df_lagged["ISR"]
X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(X_rf, y_rf, test_size=0.2, random_state=42)
X_train_rf

randomForrestmodel = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    max_features="sqrt",
)
decisionModel = DecisionTreeRegressor(
    max_depth=10,
    criterion="squared_error",
    random_state=42

)
randomForrestmodel.fit(X_train_rf, y_train_rf)
rf_pred = randomForrestmodel.predict(X_test_rf)

print("Random Forest RMSE:",
      np.sqrt(mean_squared_error(y_test_rf, rf_pred)))

importances = pd.Series(
    randomForrestmodel.feature_importances_,
    index=X_rf.columns
).sort_values(ascending=False)

print(importances)


Random Forest RMSE: 0.2371832278065893
C-Peptide        0.417227
Time (min)       0.207724
CPeptide_lag1    0.138480
CPeptide_lag2    0.125296
CPeptide_lag3    0.111273
dtype: float64


In [14]:
df_lagged["ISR"].min(), df_lagged["ISR"].max()


(np.float64(-0.352473695883384), np.float64(1.340563192167483))

In [13]:
df_lagged

,Sample ID,Sample,Gender,Age (yr),Height (cm),Weight (kg),BSA (m2),BMI (kg/m2),Group,Time (min),C-Peptide,ISR,CPeptide_lag1,CPeptide_lag2,CPeptide_lag3
3,1,302-003-5,2,41,161.5,105.9,2.079,40.6,1,4,0.94,0.169623,0.91,0.89,0.92
4,1,302-003-5,2,41,161.5,105.9,2.079,40.6,1,6,0.99,0.183886,0.94,0.91,0.89
5,1,302-003-5,2,41,161.5,105.9,2.079,40.6,1,10,1.05,0.200186,0.99,0.94,0.91
6,1,302-003-5,2,41,161.5,105.9,2.079,40.6,1,25,1.29,0.218524,1.05,0.99,0.94
7,1,302-003-5,2,41,161.5,105.9,2.079,40.6,1,30,1.24,0.191017,1.29,1.05,0.99
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1057,59,NDC_27,1,63,174.2,97.5,2.120,32.1,2,60,1.36,0.677901,1.26,1.24,1.07
1058,59,NDC_27,1,63,174.2,97.5,2.120,32.1,2,62,5.45,0.825531,1.36,1.26,1.24
1059,59,NDC_27,1,63,174.2,97.5,2.120,32.1,2,64,4.00,0.855425,5.45,1.36,1.26
1060,59,NDC_27,1,63,174.2,97.5,2.120,32.1,2,66,3.55,0.768042,4.00,5.45,1.36


In [4]:
df_lagged.columns.unique()

Index(['PatientID', 'TimeIndex', 'TimeMinutes', 'CPeptide', 'ISR',
       'CPeptide_lag1', 'CPeptide_lag2', 'CPeptide_lag3'],
      dtype='object')

In [14]:
xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

xgb_model.fit(X_train_rf, y_train_rf)
xgb_pred = xgb_model.predict(X_test_rf)

print("XGBoost RMSE:",
      np.sqrt(mean_squared_error(y_test_rf, xgb_pred)))

XGBoost RMSE: 0.32400568785506956
